# Clase 4 — De fuentes dispersas a datos integrados

Caso conductor: una plataforma de vehiculos integra SQLite y un CSV de
precios de mercado. El resultado se persiste en SQLite para que otros
procesos puedan consultarlo.

La salida importante no es una grafica: es un proceso reproducible que se
ejecuta con `python etl.py`.

## 1) El problema: los datos estan distribuidos

La plataforma de vehiculos no tiene un unico archivo analitico:

- SQLite: `vehicles`, `customers`, `sales`
- CSV: precios obtenidos de publicaciones externas
- JSON: reglas de homologacion de marcas

La pregunta de la clase es:

> ¿Como construimos un proceso repetible que obtenga, combine, valide y
> almacene esta informacion?

El flujo sera:

`Fuentes → Extract → Transform → Validate → Load → Verify`

No entrenaremos modelos en esta sesion. Construiremos los datos que otros
procesos podran consultar.

## 2) SQLite: persistencia relacional simple

La base `data/vehicles.db` contiene:

- `vehicles(vehicle_id, brand, model, year, mileage)`
- `customers(customer_id, name, city)`
- `sales(sale_id, vehicle_id, customer_id, sale_date, sale_price)`

La relacion es:

`CUSTOMERS 1 → N SALES N ← 1 VEHICLES`

SQLite permite practicar persistencia relacional en un archivo, sin administrar
un servidor de base de datos.

In [16]:
import importlib
import sqlite3
import sys
from pathlib import Path

import pandas as pd

PROJECT_DIR = Path.cwd() / 'proyecto_etl_vehiculos'
sys.path.insert(0, str(PROJECT_DIR))

import etl
import seed_database
importlib.reload(etl)
importlib.reload(seed_database)

from seed_database import seed_database
from etl import load_config, extract, transform, validate, load, configure_logging

seed_database()
config = load_config()
configure_logging(config.log_path)
print('ETL module:', etl.__file__)
print('SQLite:', config.database_path)
print('CSV:', config.market_prices_path)
print('JSON:', config.vehicle_specs_path)

SQLite listo: /Users/alder.lopez/Documents/ClasesTec/TC3009C.602/CodigoClases/Clase04/proyecto_etl_vehiculos/data/vehicles.db
vehicles: 10 | customers: 3 | sales: 3
ETL module: /Users/alder.lopez/Documents/ClasesTec/TC3009C.602/CodigoClases/Clase04/proyecto_etl_vehiculos/etl.py
SQLite: /Users/alder.lopez/Documents/ClasesTec/TC3009C.602/CodigoClases/Clase04/proyecto_etl_vehiculos/data/vehicles.db
CSV: /Users/alder.lopez/Documents/ClasesTec/TC3009C.602/CodigoClases/Clase04/proyecto_etl_vehiculos/data/market_prices.csv
JSON: /Users/alder.lopez/Documents/ClasesTec/TC3009C.602/CodigoClases/Clase04/proyecto_etl_vehiculos/data/vehicle_specs.json


## 3) SQL: consultar, relacionar y resumir

Tres operaciones sostienen gran parte del flujo:

- `WHERE` selecciona registros.
- `JOIN` relaciona tablas.
- `GROUP BY` resume y agrega.

Ejemplo: precio promedio de venta por marca.

In [17]:
with sqlite3.connect(config.database_path) as conn:
    recent = pd.read_sql(
        'SELECT brand, model, year, mileage FROM vehicles WHERE year >= 2022',
        conn,
    )
    sold_by_brand = pd.read_sql(
        '''
        SELECT v.brand, AVG(s.sale_price) AS avg_price
        FROM vehicles v
        JOIN sales s ON v.vehicle_id = s.vehicle_id
        GROUP BY v.brand
        ORDER BY avg_price DESC
        ''',
        conn,
    )

print('Vehiculos desde 2022:')
display(recent)
print('Precio promedio de venta por marca:')
display(sold_by_brand)

Vehiculos desde 2022:


,brand,model,year,mileage
0,Volkswagen,Jetta,2022,38500
1,Nissan,Sentra,2022,31000
2,Mazda,CX-5,2022,28000
3,Toyota,Corolla,2023,19000


Precio promedio de venta por marca:


,brand,avg_price
0,Toyota,385000.0
1,Volkswagen,335000.0
2,Chevrolet,220000.0


## 4) ETL: construir el dato que necesitamos

El ETL no es "limpiar un CSV". Es definir un proceso que mueve informacion
entre fuentes y un destino:

- **Extract**: SQLite + CSV + JSON.
- **Transform**: homologar, combinar, convertir tipos y crear `vehicle_age`.
- **Validate**: revisar identidad, valores y unicidad.
- **Load**: guardar en `vehicles_integrated` y separar `etl_rejects`.

La integracion es semantica: un `vehicle_id` existente no basta si marca y
modelo no coinciden.

In [18]:
import uuid

run_id = str(uuid.uuid4())
vehicles, market_prices, vehicle_specs, aliases = extract(config)
processed, rejects = transform(
    vehicles,
    market_prices,
    vehicle_specs,
    aliases,
    config.run_date,
    run_id,
)
processed.attrs.update(
    source_vehicles=len(vehicles),
    source_market_prices=len(market_prices),
    source_specs=len(vehicle_specs),
)

print('Reglas de homologacion:', aliases)
print('Especificaciones JSON:', len(vehicle_specs))
print('Columnas de salida:', processed.columns.tolist())
print('Integrados:', len(processed))
print('Rechazados:', len(rejects))
display(rejects[['vehicle_id', 'brand_market', 'model_market', 'rejection_reason']])

[13:56:24] INFO SQLite: 10 records
[13:56:24] INFO CSV: 12 records
[13:56:24] INFO JSON specs: 9 records
[13:56:24] INFO JSON aliases: 7 rules
[13:56:24] INFO Integrated: 9 records
[13:56:24] INFO Rejected: 3 records


Reglas de homologacion: {'VW': 'Volkswagen', 'VOLKSWAGEN': 'Volkswagen', 'NISSAN': 'Nissan', 'Nissan Motor': 'Nissan', 'TOYOTA': 'Toyota', 'CHEVROLET': 'Chevrolet', 'MAZDA': 'Mazda'}
Especificaciones JSON: 9
Columnas de salida: ['vehicle_id', 'brand', 'model', 'year', 'mileage', 'vehicle_age', 'city', 'category', 'market_price', 'listings_count', 'sources', 'etl_run_id', 'etl_run_date']
Integrados: 9
Rechazados: 3


,vehicle_id,brand_market,model_market,rejection_reason
9,10,Marca inexistente,Modelo Desconocido,vehicle_identity_mismatch;vehicle_specs_not_found
10,11,Nissan,Sentra,vehicle_id_not_found;vehicle_specs_not_found;p...
11,12,Toyota,Corolla,vehicle_id_not_found;vehicle_specs_not_found


## 5) Validacion, persistencia e idempotencia

La validacion detiene el proceso si la salida no cumple el contrato esperado.

La carga usa `if_exists='replace'`: para este mini ETL, volver a ejecutar el
mismo proceso reconstruye las tablas de salida en lugar de insertar duplicados.
En sistemas mayores se puede usar una clave de corrida o `UPSERT`, pero la
idea es la misma: la misma entrada produce un estado consistente.

In [19]:
validate(processed)
load(
    config,
    processed,
    rejects,
    run_id,
    pd.Timestamp.now('UTC').isoformat(),
    pd.Timestamp.now('UTC').isoformat(),
)

with sqlite3.connect(config.database_path) as conn:
    integrated_check = pd.read_sql(
        'SELECT * FROM vehicles_integrated ORDER BY vehicle_id LIMIT 10',
        conn,
    )
    rejects_check = pd.read_sql('SELECT * FROM etl_rejects', conn)
    runs_check = pd.read_sql('SELECT * FROM etl_runs ORDER BY started_at DESC LIMIT 3', conn)

print('vehicles_integrated:')
display(integrated_check)
print('etl_rejects:')
display(rejects_check[['vehicle_id', 'rejection_reason']])
print('etl_runs:')
display(runs_check)

[13:56:24] INFO Validation passed: 9 records
[13:56:24] INFO Load completed: vehicles_integrated, etl_rejects and etl_runs


vehicles_integrated:


,vehicle_id,brand,model,year,mileage,vehicle_age,city,category,market_price,listings_count,sources,etl_run_id,etl_run_date
0,1,Volkswagen,Jetta,2022.0,38500.0,4,Monterrey,Sedan,342000.0,1,marketplace_a,d83739bf-2e98-446f-a8b0-17238c02d968,2026-08-20
1,2,Nissan,Sentra,2021.0,42000.0,5,Guadalajara,Sedan,318500.0,1,marketplace_a,d83739bf-2e98-446f-a8b0-17238c02d968,2026-08-20
2,3,Chevrolet,Aveo,2020.0,68000.0,6,Ciudad de Mexico,Sedan,224000.0,1,marketplace_b,d83739bf-2e98-446f-a8b0-17238c02d968,2026-08-20
3,4,Volkswagen,Jetta,2020.0,72000.0,6,Monterrey,Sedan,355000.0,1,marketplace_b,d83739bf-2e98-446f-a8b0-17238c02d968,2026-08-20
4,5,Nissan,Sentra,2022.0,31000.0,4,Guadalajara,Sedan,325000.0,1,marketplace_b,d83739bf-2e98-446f-a8b0-17238c02d968,2026-08-20
5,6,Toyota,Corolla,2021.0,51000.0,5,Ciudad de Mexico,Sedan,390000.0,1,marketplace_a,d83739bf-2e98-446f-a8b0-17238c02d968,2026-08-20
6,7,Nissan,Versa,2020.0,59000.0,6,Monterrey,Sedan,285000.0,1,marketplace_c,d83739bf-2e98-446f-a8b0-17238c02d968,2026-08-20
7,8,Mazda,CX-5,2022.0,28000.0,4,Ciudad de Mexico,SUV,498000.0,1,marketplace_a,d83739bf-2e98-446f-a8b0-17238c02d968,2026-08-20
8,9,Chevrolet,Aveo,2019.0,83000.0,7,Guadalajara,Sedan,218000.0,1,marketplace_c,d83739bf-2e98-446f-a8b0-17238c02d968,2026-08-20


etl_rejects:


,vehicle_id,rejection_reason
0,10,vehicle_identity_mismatch;vehicle_specs_not_found
1,11,vehicle_id_not_found;vehicle_specs_not_found;p...
2,12,vehicle_id_not_found;vehicle_specs_not_found


etl_runs:


,run_id,started_at,finished_at,source_vehicles,source_market_prices,source_specs,integrated_rows,rejected_rows,status
0,d83739bf-2e98-446f-a8b0-17238c02d968,2026-08-20T19:56:24.946035+00:00,2026-08-20T19:56:24.957019+00:00,10,12,9,9,3,success
1,d3b44125-1f2f-4ae9-92b4-8d0d5b836ece,2026-08-20T18:53:53.302744+00:00,2026-08-20T18:53:53.308159+00:00,10,12,9,9,3,success
2,aea34993-290d-4843-a216-82e5cce943b4,2026-08-20T18:53:11.871480+00:00,2026-08-20T18:53:11.911054+00:00,10,12,9,9,3,success


## 6) Del notebook a un proceso ejecutable

El notebook sirve para explorar y explicar. El proceso operativo vive en
`etl.py` y tiene un unico punto de entrada:

```bash
python etl.py
```

Para reproducir todo desde cero:

```bash
./run_etl.sh
```

Ese comando prepara la base, ejecuta Extract → Transform → Validate → Load y
consulta el resultado. Puede programarse con `cron` en macOS/Linux o Task
Scheduler en Windows.

Requisitos minimos de operacion:

- configuracion centralizada;
- logging de inicio, volumenes, rechazados y finalizacion;
- errores visibles, no silenciosos;
- idempotencia basica.

## 7) Arquitectura resultante

```text
SQLite + CSV + JSON
        ↓
      EXTRACT
        ↓
     TRANSFORM
  Integracion + Homologacion + Reglas
        ↓
     VALIDATE
        ↓
       LOAD
        ↓
SQLite: vehicles_integrated / etl_rejects
        ↑
Scheduler: cron o Task Scheduler
```

La capacidad nueva de esta clase es que el dato integrado deja de depender de
una persona que abre y ejecuta celdas manualmente.

In [20]:
counts = pd.DataFrame(
    [
        {'table_name': 'vehicles_sqlite', 'rows': len(vehicles)},
        {'table_name': 'market_prices_csv', 'rows': len(market_prices)},
        {'table_name': 'vehicle_specs_json', 'rows': len(vehicle_specs)},
        {'table_name': 'vehicles_integrated', 'rows': len(processed)},
        {'table_name': 'etl_rejects', 'rows': len(rejects)},
    ]
)

display(counts)
print('La corrida queda auditada en etl_runs con origenes, integrados, rechazados y estado.')
print('Si se vuelve a ejecutar el ETL, vehicles_integrated se reemplaza de forma transaccional.')

,table_name,rows
0,vehicles_sqlite,10
1,market_prices_csv,12
2,vehicle_specs_json,9
3,vehicles_integrated,9
4,etl_rejects,3


La corrida queda auditada en etl_runs con origenes, integrados, rechazados y estado.
Si se vuelve a ejecutar el ETL, vehicles_integrated se reemplaza de forma transaccional.


## 8) Actividad practica: ampliar el ETL

A partir de este proyecto, agregar una fuente `vehicle_reviews.json` con:

- `vehicle_id`
- `rating`
- `review_date`

El objetivo no es agregar una pantalla ni un modelo. Es extender el flujo de
datos.

### Requisitos

1. **Extract**: leer el JSON y validar sus columnas.
2. **Transform**: calcular `avg_rating` por `vehicle_id`.
3. **Integrate**: unirlo con `vehicles_integrated`.
4. **Validate**: aceptar solo ratings entre 1 y 5.
5. **Load**: guardar `vehicles_enriched`.
6. **Audit**: registrar cuantas filas fueron aceptadas y rechazadas.
7. **Idempotence**: ejecutar dos veces y comprobar que no hay duplicados.

### Criterios de aceptacion

- `python etl.py` sigue siendo el punto de entrada.
- Una fila invalida no desaparece: aparece en una tabla de rechazos.
- La salida tiene una clave unica por `vehicle_id`.
- La segunda ejecucion deja el mismo numero de filas que la primera.
- El log indica inicio, fuentes, integrados, rechazados, carga y finalizacion.

La pregunta que debe contestar el estudiante es: **¿que contrato debe cumplir
cada fuente para que el destino sea confiable?**

## 9) Primera experiencia visual con Duckle

La implementación en Python muestra cómo programar el ETL. Antes de construir
el caso con dos fuentes, empezamos con un ejemplo aislado para entender qué
hace Duckle y para qué sirve cada tipo de nodo.

### Práctica 0: CSV → Filter → CSV

Usa `practica_duckle/README_INICIAL.md`.

```text
vehicle_listings_demo.csv
          ↓
 Filter: status = 'active'
          ↓
 vehicle_listings_active.csv
```

Resultado esperado:

```text
6 filas de entrada → 4 filas de salida
```

El estudiante debe observar el preview, ejecutar el pipeline, abrir el Plan
para ver el SQL y ejecutar una segunda vez con `overwrite`. La idea central:

> Duckle representa un proceso de datos como un grafo ejecutable: una fuente
> entrega filas, una transformación decide qué cambia y un sink persiste el
> resultado.

Conceptualmente, Duckle genera algo equivalente a:

```sql
COPY (
    SELECT *
    FROM read_csv_auto('data/vehicle_listings_demo.csv')
    WHERE status = 'active'
) TO 'data/vehicle_listings_active.csv' (HEADER, OVERWRITE);
```

La equivalencia es:

```text
Notebook: pandas.read_csv → filtro → to_csv
Duckle:   CSV source      → Filter → CSV sink
```

Después de entender esta tubería mínima, se pasa al caso de vehículos con
SQLite + CSV, donde aparecen integración, homologación y rejects.

La actividad posterior agrega `vehicle_specs.json`, calcula una segunda
integración y exige justificar el tipo de join, los rechazos y la idempotencia.

### ¿Qué ocurre dentro del notebook?

Duckle es una aplicación visual externa, no una librería que se ejecute
automáticamente dentro del kernel de Jupyter. Por eso el notebook no puede
mostrar el canvas de Duckle en una celda.

La demostración se divide en dos partes:

1. Esta celda ejecuta el flujo equivalente con Python para observar el resultado.
2. `practica_duckle/README_INICIAL.md` indica cómo construir el mismo flujo
   visualmente en Duckle y revisar el SQL que genera.

La equivalencia es:

```text
Notebook: pandas.read_csv → filtro → to_csv
Duckle:   CSV source      → Filter → CSV sink
```

El resultado debe ser el mismo: 6 filas de entrada y 4 filas activas.

In [21]:
from pathlib import Path

practice_dir = Path.cwd() / 'practica_duckle'
input_path = practice_dir / 'data' / 'vehicle_listings_demo.csv'
output_path = practice_dir / 'data' / 'vehicle_listings_active_from_notebook.csv'

listings = pd.read_csv(input_path)
active_listings = listings.loc[listings['status'].eq('active')].copy()
active_listings.to_csv(output_path, index=False)

print(f'Entrada: {len(listings)} filas')
print("Transformacion: status == 'active'")
print(f'Salida: {len(active_listings)} filas')
print(f'Archivo: {output_path}')
display(active_listings)

Entrada: 6 filas
Transformacion: status == 'active'
Salida: 4 filas
Archivo: /Users/alder.lopez/Documents/ClasesTec/TC3009C.602/CodigoClases/Clase04/practica_duckle/data/vehicle_listings_active_from_notebook.csv


,vehicle_id,brand,model,year,price,status
0,1,Nissan,Sentra,2022,318500,active
1,2,VW,Jetta,2021,342000,active
3,4,Chevrolet,Aveo,2019,218000,active
4,5,Toyota,Corolla,2023,390000,active
